In [1]:
from pyspark.sql import SparkSession
import duckdb
import pandas as pd
import os
import sys
spark = SparkSession.builder \
    .appName('NetworkIntrusionETL_Load') \
    .config('spark.driver.memory', '4g') \
    .getOrCreate()

spark.sparkContext.setLogLevel('WARN')

STAGING  = os.path.abspath(os.path.join(os.getcwd(), '..', 'Data', 'staging')).replace('\\', '/')
db_path  = os.path.abspath(os.path.join(os.getcwd(), '..', 'duck_db', 'network_intrusion.duckdb'))
print('Staging dir:', STAGING)
print('DuckDB path:', db_path)

Staging dir: d:/Projects/Big Data/BG  Assignmnet/Assignmnet/Data/staging
DuckDB path: d:\Projects\Big Data\BG  Assignmnet\Assignmnet\duck_db\network_intrusion.duckdb


In [2]:
# ── Read transformed parquet with PySpark ────────────────────────────────────
unified_df = spark.read.parquet(STAGING + '/unified_transformed')
summary_df = spark.read.parquet(STAGING + '/summary_agg')

print('unified rows:', unified_df.count())
print('summary rows:', summary_df.count())

unified rows: 2732486
summary rows: 29


In [3]:
# ── Convert to pandas for DuckDB ingestion ────────────────────────────────────
unified_pd = unified_df.toPandas()
summary_pd = summary_df.toPandas()

unified_pd = unified_pd.where(pd.notnull(unified_pd), None)
summary_pd = summary_pd.where(pd.notnull(summary_pd), None)

print('Converted to pandas')

Converted to pandas


In [4]:
# ── Connect to DuckDB and run setup SQL ───────────────────────────────────────
con = duckdb.connect(db_path)

setup_sql_path = os.path.abspath(os.path.join(os.getcwd(), '..', 'duck_db', 'setup.sql'))
with open(setup_sql_path, 'r') as f:
    setup_sql = f.read()

for stmt in setup_sql.split(';'):
    s = stmt.strip()
    if s:
        con.execute(s)

print('Schema ready')

Schema ready


In [5]:
# ── Truncate and reload ───────────────────────────────────────────────────────
con.execute('DELETE FROM network_traffic')
con.execute('DELETE FROM attack_summary')

con.register('unified_pd', unified_pd)
con.execute('''
    INSERT INTO network_traffic (
        dst_port, duration, src_pkts, flow_bytes_per_sec,
        flow_pkts_per_sec, mean_pkt_len, std_pkt_len,
        fin_flag, psh_flag, ack_flag,
        attack_category, is_attack, source_dataset
    )
    SELECT
        dst_port, duration, src_pkts, flow_bytes_per_sec,
        flow_pkts_per_sec, mean_pkt_len, std_pkt_len,
        fin_flag, psh_flag, ack_flag,
        attack_category, is_attack, source_dataset
    FROM unified_pd
''')
print('network_traffic inserted:', len(unified_pd))

con.register('summary_pd', summary_pd)
con.execute('''
    INSERT INTO attack_summary (
        source_dataset, attack_category, is_attack, record_count,
        avg_duration, avg_flow_bytes_per_sec, avg_flow_pkts_per_sec,
        avg_mean_pkt_len, avg_src_pkts
    )
    SELECT
        source_dataset, attack_category, is_attack, record_count,
        avg_duration, avg_flow_bytes_per_sec, avg_flow_pkts_per_sec,
        avg_mean_pkt_len, avg_src_pkts
    FROM summary_pd
''')
print('attack_summary inserted:', len(summary_pd))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

network_traffic inserted: 2732486
attack_summary inserted: 29


In [6]:
# ── Verify ────────────────────────────────────────────────────────────────────
print('network_traffic count:', con.execute('SELECT COUNT(*) FROM network_traffic').fetchone()[0])
print('attack_summary count: ', con.execute('SELECT COUNT(*) FROM attack_summary').fetchone()[0])

print('\nAttack distribution:')
print(con.execute('''
    SELECT source_dataset, is_attack, COUNT(*) AS cnt
    FROM network_traffic
    GROUP BY source_dataset, is_attack
    ORDER BY source_dataset, is_attack
''').df())

print('\nTop attack categories:')
print(con.execute('''
    SELECT attack_category, SUM(record_count) AS total
    FROM attack_summary
    GROUP BY attack_category
    ORDER BY total DESC
    LIMIT 10
''').df())

con.close()
print('\nLoad complete.')

network_traffic count: 2732486
attack_summary count:  29

Attack distribution:
  source_dataset  is_attack      cnt
0     CICIDS2017          1  2520751
1      UNSW_NB15          0    56000
2      UNSW_NB15          1   119341
3      ZEEK_CONN          0    36394

Top attack categories:
  attack_category      total
0  Normal Traffic  2095057.0
1             DoS   206009.0
2            DDoS   128014.0
3   Port Scanning    90694.0
4          Normal    56000.0
5         Generic    40000.0
6        Exploits    33393.0
7         Unknown    20030.0
8         Fuzzers    18184.0
9  Reconnaissance    10491.0

Load complete.
